# Stage 1: Non-instruction Causal LM Fine-tuning or Domain-Adaptive Continued Pretraining

### Pharma PDF → Raw Text → Non-Instruction Causal LLM Fine-Tuning

- Causal meaning: Past causes future prediction.

- Goal: Use a pharma-domain PDF as raw text data and perform **non-instruction causal language model fine-tuning** using LoRA/QLoRA.

## Pipeline

```text
Pharma PDF
   ↓
PDF text extraction
   ↓
Text cleaning and normalization
   ↓
Data creation
   ↓
Hugging Face Dataset Conversion
   ↓
Tokenization
   ↓
LoRA/QLoRA fine-tuning
   ↓
Validation loss
   ↓
Adapter saving and reloading
   ↓
Text continuation inference
```

## Continued Pretraining vs Instruction Fine-Tuning

In this notebook, we are performing **continued pretraining / non-instruction fine-tuning** on raw pharma PDF text.

The model is given raw domain text such as:

> Metformin is one of the most widely prescribed oral antihyperglycemic agents...

The model then learns to **predict the next token** from this raw text.

This means the model learns:

- Pharma language
- Drug names
- Medical terminology
- Scientific writing style
- Domain-specific sentence patterns

However, the model is **not explicitly taught**:

- How to answer a user's question
- How to follow instructions
- How to respond in Q&A format
- How to behave like a domain-specific chatbot

---

## What Instruction Fine-Tuning Looks Like

In instruction fine-tuning, the training data is prepared in an **instruction-response format**.

### Example

```json
{
  "instruction": "Explain the mechanism of action of Metformin.",
  "input": "",
  "output": "Metformin primarily activates AMPK, which improves glucose uptake and reduces hepatic gluconeogenesis."
}
```

```json
{
  "messages": [
    {
      "role": "user",
      "content": "What is the primary mechanism of action of Metformin?"
    },
    {
      "role": "assistant",
      "content": "Metformin primarily works by activating AMPK..."
    }
  ]
}
```

## Complete Pipeline with all Three FT Techniques:

```text
Non-instrcution FT(RAW Data)
      ↓
will save the model
      ↓
load the model
      ↓
I will perform instruction FT on same model(question/answer data)
      ↓
will save our model
      ↓
again will load the same model
      ↓
will perform the preference tuning on top of it(choosed/ reject data)
   
```

we are going to train LORA adapter

In [2]:
# ============================================================
# 1. Install required libraries
# ============================================================
# PyMuPDF: PDF text extraction
# datasets: Hugging Face dataset creation
# transformers/accelerate: model, tokenizer, Trainer
# peft: LoRA/QLoRA adapters
# bitsandbytes: 4-bit/8-bit quantized loading


In [3]:
!pip install "torchao>=0.16.0"

In [4]:
!pip install -q -U pymupdf datasets transformers accelerate peft bitsandbytes sentencepiece

In [5]:
# To ignore warnings
import warnings
warnings.filterwarnings("ignore")

In [6]:
# ============================================================
# 2. Imports
# ============================================================
import os
import re
import gc
import math
import json
import random
import unicodedata
from dataclasses import dataclass, asdict
from typing import List, Dict, Any

import fitz  # PyMuPDF
import torch
from datasets import Dataset, DatasetDict

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    set_seed,
)

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
)

In [7]:
# ============================================================
# 3. Global configuration
# ============================================================
# Keep all important parameters in one place.
# This makes the notebook easier to debug, reproduce, and productionize.

from dataclasses import dataclass, asdict

@dataclass
class Config:
    # Path of the pharma PDF file that will be used as the raw domain corpus.
    pdf_path: str = "/content/Metformin-Lipid-Therapy-Knowledge.pdf"

    # Base causal language model that we will fine-tune on pharma-domain text.
    model_name: str = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

    # Directory where training checkpoints will be saved during fine-tuning.
    output_dir: str = "/content/pharma_tinyllama_lora_output"

    # Directory where the final trained LoRA adapter will be saved.
    adapter_dir: str = "/content/pharma_tinyllama_lora_adapter"

    # Directory where cleaned and processed training data will be saved.
    processed_data_dir: str = "/content/pharma_processed_data"

    # Minimum paragraph length required to keep a paragraph for training.
    min_chars_per_paragraph: int = 80

    # Number of tokens in each training block for causal language modeling.
    block_size: int = 512

    # Percentage of data used for validation instead of training.
    test_size: float = 0.15

    # Random seed used to make splitting and training more reproducible.
    seed: int = 42

    # LoRA rank; controls the size and capacity of the trainable adapter.
    lora_r: int = 16

    # LoRA scaling factor; controls the strength of the LoRA update.
    lora_alpha: int = 32

    # Dropout applied inside LoRA layers to reduce overfitting.
    lora_dropout: float = 0.05

    # Number of times the model will see the complete training dataset.
    num_train_epochs: float = 10.0

    # Number of training samples processed per GPU/device at one time.
    per_device_train_batch_size: int = 1

    # Number of validation samples processed per GPU/device at one time.
    per_device_eval_batch_size: int = 1

    # Number of small batches accumulated before one optimizer update.
    gradient_accumulation_steps: int = 8

    # Step size used by the optimizer to update trainable LoRA weights.
    learning_rate: float = 2e-4

    # Fraction of early training steps used to gradually increase learning rate.
    warmup_ratio: float = 0.03

    # Regularization value used to prevent weights from becoming too large.
    weight_decay: float = 0.01

    # Number of training steps after which logs will be printed.
    logging_steps=1
    logging_first_step=True

    # Number of training steps after which validation will be performed.
    eval_steps: int = 10

    # Number of training steps after which a checkpoint will be saved.
    save_steps: int = 25

    # Maximum number of checkpoints to keep; older checkpoints will be deleted.
    save_total_limit: int = 2

    # Maximum number of training steps; -1 means train using num_train_epochs.
    max_steps: int = -1

In [8]:
config = Config()
config

Config(pdf_path='/content/Metformin-Lipid-Therapy-Knowledge.pdf', model_name='TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T', output_dir='/content/pharma_tinyllama_lora_output', adapter_dir='/content/pharma_tinyllama_lora_adapter', processed_data_dir='/content/pharma_processed_data', min_chars_per_paragraph=80, block_size=512, test_size=0.15, seed=42, lora_r=16, lora_alpha=32, lora_dropout=0.05, num_train_epochs=10.0, per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_accumulation_steps=8, learning_rate=0.0002, warmup_ratio=0.03, weight_decay=0.01, eval_steps=10, save_steps=25, save_total_limit=2, max_steps=-1)

In [9]:
# To check values of all the configs in json format
print(json.dumps(asdict(config), indent=2))

{
  "pdf_path": "/content/Metformin-Lipid-Therapy-Knowledge.pdf",
  "model_name": "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T",
  "output_dir": "/content/pharma_tinyllama_lora_output",
  "adapter_dir": "/content/pharma_tinyllama_lora_adapter",
  "processed_data_dir": "/content/pharma_processed_data",
  "min_chars_per_paragraph": 80,
  "block_size": 512,
  "test_size": 0.15,
  "seed": 42,
  "lora_r": 16,
  "lora_alpha": 32,
  "lora_dropout": 0.05,
  "num_train_epochs": 10.0,
  "per_device_train_batch_size": 1,
  "per_device_eval_batch_size": 1,
  "gradient_accumulation_steps": 8,
  "learning_rate": 0.0002,
  "warmup_ratio": 0.03,
  "weight_decay": 0.01,
  "eval_steps": 10,
  "save_steps": 25,
  "save_total_limit": 2,
  "max_steps": -1
}


In [10]:
# To check value of any parameter
config.output_dir

'/content/pharma_tinyllama_lora_output'

In [11]:
# Code for creating directory
os.makedirs(config.output_dir, exist_ok=True)
os.makedirs(config.adapter_dir, exist_ok=True)
os.makedirs(config.processed_data_dir, exist_ok=True)

In [12]:
# ============================================================
# 4. Optional Colab upload helper
# ============================================================
# Run this cell only if your PDF is not already present at config.pdf_path.

if not os.path.exists(config.pdf_path):
    print(f"PDF not found at: {config.pdf_path}")
else:
    print(f"PDF found: {config.pdf_path}")

PDF found: /content/Metformin-Lipid-Therapy-Knowledge.pdf


## Uploading to HuggingFace

In [13]:
# ============================================================
# Hugging Face repo names
# ============================================================

HF_USERNAME = "AreebAhmxd"

BASE_MODEL_NAME = config.model_name

# Stage 1: Non-instruction LoRA adapter
HF_REPO_NON_INSTRUCTION_ADAPTER = f"{HF_USERNAME}/pharma-tinyllama-1.1b-non-instruction-lora-adapter"

# Stage 1 merged model
HF_REPO_NON_INSTRUCTION_MERGED = f"{HF_USERNAME}/pharma-tinyllama-1.1b-non-instruction-model"

# Stage 2: Instruction LoRA adapter
HF_REPO_INSTRUCTION_ADAPTER = f"{HF_USERNAME}/pharma-tinyllama-1.1b-instruction-lora-adapter"

# Stage 2 merged model
HF_REPO_INSTRUCTION_MERGED = f"{HF_USERNAME}/pharma-tinyllama-1.1b-instruction-model"

# Stage 3: DPO preference LoRA adapter
HF_REPO_DPO_ADAPTER = f"{HF_USERNAME}/pharma-tinyllama-1.1b-dpo-lora-adapter"

# Stage 3 final merged model
HF_REPO_DPO_MERGED = f"{HF_USERNAME}/pharma-tinyllama-1.1b-dpo-model"

print(HF_REPO_NON_INSTRUCTION_ADAPTER)
print(HF_REPO_INSTRUCTION_ADAPTER)
print(HF_REPO_DPO_ADAPTER)

AreebAhmxd/pharma-tinyllama-1.1b-non-instruction-lora-adapter
AreebAhmxd/pharma-tinyllama-1.1b-instruction-lora-adapter
AreebAhmxd/pharma-tinyllama-1.1b-dpo-lora-adapter


### Extract Text from PDF (Data Parsing)
- Library used to parse the data: PyMuPDF

In [14]:
# # ============================================================
# # 5. Extract text from PDF
# # ============================================================
def extract_pdf_pages(pdf_path: str) -> List[Dict[str, Any]]:
    # Extract page-level text from a PDF.
    pages = []
    with fitz.open(pdf_path) as doc:
        for page_index, page in enumerate(doc, start=1):
            text = page.get_text("text")
            text = text.strip() if text else ""
            if text:
                pages.append({
                    "page": page_index,
                    "text": text,
                    "char_count": len(text),
                })
    return pages


In [15]:
pdf_pages = extract_pdf_pages(config.pdf_path)

In [16]:
print(f"Total pages with extracted text: {len(pdf_pages)}")
print("Page-level character counts:")
for item in pdf_pages:
    print(f"Page {item['page']}: {item['char_count']} characters")

Total pages with extracted text: 6
Page-level character counts:
Page 1: 2244 characters
Page 2: 2889 characters
Page 3: 2636 characters
Page 4: 2416 characters
Page 5: 2613 characters
Page 6: 2761 characters


In [17]:
# check 1st page
print(pdf_pages[0]["text"])

Metformin is one of the most widely prescribed oral antihyperglycemic agents.​
 Its primary mechanism of action involves the activation of AMP-activated protein kinase 
(AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation 
while inhibiting hepatic gluconeogenesis.​
 Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes 
and display anti-inflammatory properties.​
 Recent studies also suggest potential anticancer effects through inhibition of the mTOR 
signaling pathway and suppression of tumor angiogenesis. 
 
Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in 
significant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to 
monotherapy.​
 Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal 
wall, reducing cholesterol absorption, while Atorvastatin inhibits hepatic HMG-CoA reductase, 
suppressing endogenous cho

### Text / Data Cleaning

| Cleaning Step                          | Code / Logic                             | What It Does                                                                  | Example Before                                                  | Example After                                                  | Why It Matters for Fine-Tuning                                            |
| -------------------------------------- | ---------------------------------------- | ----------------------------------------------------------------------------- | --------------------------------------------------------------- | -------------------------------------------------------------- | ------------------------------------------------------------------------- |
| Unicode normalization                  | `unicodedata.normalize("NFKC", text)`    | Converts unusual Unicode characters into standard readable characters.        | `ＡＭＰＫ`, `ﬁ`                                                     | `AMPK`, `fi`                                                   | Prevents tokenizer confusion caused by hidden or non-standard characters. |
| Remove zero-width characters           | `text.replace("\u200b", "")`             | Removes invisible zero-width spaces from PDF text.                            | `Metformin​ activates AMPK`                                     | `Metformin activates AMPK`                                     | Invisible characters can create bad tokens and noisy training data.       |
| Remove BOM / hidden marker             | `text.replace("\ufeff", "")`             | Removes hidden Byte Order Mark characters sometimes found in extracted text.  | `﻿Metformin is used...`                                         | `Metformin is used...`                                         | Keeps the training text clean and consistent.                             |
| Fix hyphenated line breaks             | `re.sub(r"(\w)-\n(\w)", r"\1\2", text)`  | Joins words that were broken across PDF lines.                                | `gluconeogene-\nsis`                                            | `gluconeogenesis`                                              | Prevents the model from learning broken medical terms.                    |
| Normalize spaces and tabs              | `re.sub(r"[ \t]+", " ", text)`           | Converts multiple spaces or tabs into one space.                              | `Metformin     activates    AMPK`                               | `Metformin activates AMPK`                                     | Makes text consistent and easier for tokenizer/model to learn.            |
| Normalize blank lines                  | `re.sub(r"\n{3,}", "\n\n", text)`        | Converts too many blank lines into a proper paragraph gap.                    | `Para 1\n\n\n\nPara 2`                                          | `Para 1\n\nPara 2`                                             | Preserves paragraph structure without unnecessary whitespace noise.       |
| Remove standalone page numbers         | `re.sub(r"(?m)^\s*\d+\s*$", "", text)`   | Removes lines that contain only page numbers.                                 | `1` or `23`                                                     | Removed                                                        | Prevents the model from learning irrelevant PDF page numbers.             |
| Split into paragraphs                  | `re.split(r"\n\s*\n", text)`             | Splits text wherever there is a blank line.                                   | `Para 1\n\nPara 2`                                              | `["Para 1", "Para 2"]`                                         | Helps preserve meaningful document structure.                             |
| Remove line wrapping inside paragraphs | `re.sub(r"\n+", " ", paragraph)`         | Converts broken lines inside the same paragraph into a single paragraph line. | `Metformin is widely prescribed\noral antihyperglycemic agent.` | `Metformin is widely prescribed oral antihyperglycemic agent.` | Prevents the model from learning artificial PDF line breaks.              |
| Normalize paragraph spacing            | `re.sub(r"\s+", " ", paragraph).strip()` | Removes extra spaces inside each paragraph and trims start/end spaces.        | `  Metformin   activates   AMPK.  `                             | `Metformin activates AMPK.`                                    | Produces clean, readable training examples.                               |
| Remove empty paragraphs                | `if paragraph:`                          | Keeps only non-empty cleaned paragraphs.                                      | `""`                                                            | Removed                                                        | Avoids useless blank samples in the dataset.                              |
| Rebuild cleaned text                   | `"\n\n".join(cleaned_paragraphs)`        | Joins cleaned paragraphs with two newlines.                                   | List of cleaned paragraphs                                      | Clean paragraph-level text                                     | Creates a clean corpus suitable for causal LM training.                   |
| Track cleaned page length              | `char_count: len(cleaned_text)`          | Stores number of characters after cleaning.                                   | Raw page length unknown                                         | `char_count = 1450`                                            | Helps debug whether a page has too little or too much extracted content.  |
| Preview cleaned output                 | `cleaned_pages[0]["text"][:1500]`        | Prints first 1500 characters of cleaned page 1.                               | Full cleaned page                                               | Preview text                                                   | Helps manually verify that cleaning worked correctly.                     |


In [18]:
# ============================================================
# 6. Text cleaning utilities
# ============================================================

In [19]:
def clean_pdf_text(text: str) -> str:
    # Standardize Unicode text so visually similar characters are treated consistently.
    # Example: "ＡＭＰＫ" becomes "AMPK" and "ﬁ" becomes "fi".
    text = unicodedata.normalize("NFKC", text)

    # Remove invisible characters that may appear during PDF text extraction.
    text = text.replace("\u200b", "").replace("\ufeff", "")

    # Join words broken by line hyphenation, e.g., "gluconeogene-\nsis" -> "gluconeogenesis".
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # Replace multiple spaces/tabs with a single space.
    text = re.sub(r"[ \t]+", " ", text)

    # Convert three or more newlines into a standard paragraph break.
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Remove lines that contain only page numbers.
    text = re.sub(r"(?m)^\s*\d+\s*$", "", text)

    # Split text into paragraphs, clean each paragraph, and remove empty ones.
    paragraphs = []
    for paragraph in re.split(r"\n\s*\n", text):
        paragraph = re.sub(r"\n+", " ", paragraph)
        paragraph = re.sub(r"\s+", " ", paragraph).strip()

        if paragraph:
            paragraphs.append(paragraph)

    # Join cleaned paragraphs with one blank line between them.
    return "\n\n".join(paragraphs)

In [20]:
cleaned_pages = [] # To store cleaned pages

for page in pdf_pages:
    cleaned_text = clean_pdf_text(page["text"])
    cleaned_pages.append({
        "page": page["page"],
        "text": cleaned_text,
        "char_count": len(cleaned_text),
    })

cleaned_pages

[{'page': 1,
  'text': 'Metformin is one of the most widely prescribed oral antihyperglycemic agents. Its primary mechanism of action involves the activation of AMP-activated protein kinase (AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while inhibiting hepatic gluconeogenesis. Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes and display anti-inflammatory properties. Recent studies also suggest potential anticancer effects through inhibition of the mTOR signaling pathway and suppression of tumor angiogenesis.\n\nClinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in significant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to monotherapy. Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal wall, reducing cholesterol absorption, while Atorvastatin inhibits hepatic HMG-CoA reductase, suppressing endoge

In [21]:
print("Total cleaned pages:", len(cleaned_pages))

Total cleaned pages: 6


In [22]:
# Check 1st page
print("Cleaned page preview:\n")
print(cleaned_pages[0]["text"])

Cleaned page preview:

Metformin is one of the most widely prescribed oral antihyperglycemic agents. Its primary mechanism of action involves the activation of AMP-activated protein kinase (AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while inhibiting hepatic gluconeogenesis. Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes and display anti-inflammatory properties. Recent studies also suggest potential anticancer effects through inhibition of the mTOR signaling pathway and suppression of tumor angiogenesis.

Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in significant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to monotherapy. Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal wall, reducing cholesterol absorption, while Atorvastatin inhibits hepatic HMG-CoA reductase, suppressing endogenou

### Split cleaned pages into paragraphs

In [23]:
# ============================================================
# 7. Split cleaned pages into paragraphs
# ============================================================
# This step converts cleaned page-level text into paragraph-level records.

def split_into_paragraph_records(cleaned_pages, min_chars=80):
    paragraph_records = []

    for page in cleaned_pages:
        # Split page text into paragraphs using blank lines.
        paragraphs = page["text"].split("\n\n")

        for paragraph_index, paragraph in enumerate(paragraphs, start=1):
            # Remove extra spaces from the beginning and end.
            paragraph = paragraph.strip()

            # Skip very short paragraphs because they are usually headings, page numbers, or noise.
            if len(paragraph) < min_chars:
                continue

            # Store each useful paragraph with basic metadata.
            paragraph_records.append({
                "text": paragraph,
                "source_page": page["page"],
                "paragraph_id": paragraph_index,
                "char_count": len(paragraph),
            })

    return paragraph_records

In [24]:
paragraph_records = split_into_paragraph_records(cleaned_pages)

In [25]:
# Final Data we will use for FT
print(f"Total paragraph records: {len(paragraph_records)}")
for i, record in enumerate(paragraph_records[:5]):
    print("=" * 100)
    print(f"Record {i} | Page {record['source_page']} | Characters: {record['char_count']}")
    print(record["text"])

Total paragraph records: 9
Record 0 | Page 1 | Characters: 575
Metformin is one of the most widely prescribed oral antihyperglycemic agents. Its primary mechanism of action involves the activation of AMP-activated protein kinase (AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while inhibiting hepatic gluconeogenesis. Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes and display anti-inflammatory properties. Recent studies also suggest potential anticancer effects through inhibition of the mTOR signaling pathway and suppression of tumor angiogenesis.
Record 1 | Page 1 | Characters: 598
Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in significant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to monotherapy. Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal wall, reducing cholesterol absorption, w

### Save extracted and cleaned corpus for auditability

In [26]:
# ============================================================
# 8. Save extracted and cleaned corpus for auditability
# ============================================================
# In real projects, always save intermediate datasets.
# This helps with reproducibility, debugging, and compliance review.

raw_pages_path = os.path.join(config.processed_data_dir, "pdf_pages_raw.jsonl")
paragraphs_path = os.path.join(config.processed_data_dir, "pharma_paragraph_process.jsonl")

with open(raw_pages_path, "w", encoding="utf-8") as f:
    for item in pdf_pages:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

with open(paragraphs_path, "w", encoding="utf-8") as f:
    for item in paragraph_records:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Saved raw pages to: {raw_pages_path}")
print(f"Saved cleaned paragraph corpus to: {paragraphs_path}")

Saved raw pages to: /content/pharma_processed_data/pdf_pages_raw.jsonl
Saved cleaned paragraph corpus to: /content/pharma_processed_data/pharma_paragraph_process.jsonl


### Convert Data into HF Format

In [27]:
# ============================================================
# 9. Create Hugging Face Dataset
# ============================================================

if len(paragraph_records) < 2:
    raise ValueError(
        "The extracted corpus is too small. Please provide a larger pharma PDF or lower min_chars_per_paragraph."
    )

text_dataset = Dataset.from_list(paragraph_records)


In [28]:
print(text_dataset)

Dataset({
    features: ['text', 'source_page', 'paragraph_id', 'char_count'],
    num_rows: 9
})


In [29]:
print(text_dataset[0])

{'text': 'Metformin is one of the most widely prescribed oral antihyperglycemic agents. Its primary mechanism of action involves the activation of AMP-activated protein kinase (AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while inhibiting hepatic gluconeogenesis. Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes and display anti-inflammatory properties. Recent studies also suggest potential anticancer effects through inhibition of the mTOR signaling pathway and suppression of tumor angiogenesis.', 'source_page': 1, 'paragraph_id': 1, 'char_count': 575}


### Split Dataset into Train/eval (test)

In [30]:
# ============================================================
# 10. Train/eval split
# ============================================================
# Even for small demos, keep an evaluation set.
# This gives us validation loss and perplexity.

split_dataset = text_dataset.train_test_split(test_size=config.test_size, seed=config.seed)

raw_datasets = DatasetDict({
    "train": split_dataset["train"],
    "validation": split_dataset["test"],
})

print(raw_datasets)

DatasetDict({
    train: Dataset({
        features: ['text', 'source_page', 'paragraph_id', 'char_count'],
        num_rows: 7
    })
    validation: Dataset({
        features: ['text', 'source_page', 'paragraph_id', 'char_count'],
        num_rows: 2
    })
})


## 11. Load tokenizer

- The tokenizer converts text into token IDs.

- For causal language modeling, the model learns:

```text
- Given previous tokens, predict the next token.
```

- This is why we call it **non-instruction causal LM fine-tuning**.

In [31]:
# ============================================================
# 11. Load tokenizer
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=True)

# Some Llama-style models do not define a pad token.
# For causal LM fine-tuning, using EOS as PAD is a common practical choice.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token # eos - end of sentence token

tokenizer.padding_side = "right"

In [32]:
# We are usinng this end of sentence token as padding:
tokenizer.eos_token

'</s>'

In [33]:
# Check details of tokenization Model
print(f"Tokenizer loaded: {config.model_name}")
print(f"Vocab size: {len(tokenizer)}")
print(f"Pad token: {tokenizer.pad_token} | Pad token id: {tokenizer.pad_token_id}")
print(f"EOS token: {tokenizer.eos_token} | EOS token id: {tokenizer.eos_token_id}")

Tokenizer loaded: TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T
Vocab size: 32000
Pad token: </s> | Pad token id: 2
EOS token: </s> | EOS token id: 2


### 12. Tokenization and Text Packing
### What Does `512` Mean in Text Packing?

- In this notebook, `512` means the **sequence length** or **block size** used for causal language model training.
- It is **not the embedding size**.
- It simply means:
> Each training example will contain 512 tokens.

---
## Example

Suppose the tokenizer converts our pharma text into 1,300 tokens:

```text
[token_1, token_2, token_3, ..., token_1300]?

if we set:
block_size = 512

then the tokens are split like this:
- Block 1 = token 1 to token 512
- Block 2 = token 513 to token 1024
- Remaining tokens = token 1025 to token 1300

Is 512 Padding?
Not exactly.

512 is the target length of each training block.

If we use text packing, we try to fill each block with real tokens, so padding is reduced.

Without packing:
Paragraph 1 = 100 tokens + 412 padding tokens
Paragraph 2 = 200 tokens + 312 padding tokens

With packing:
Block 1 = 512 real tokens
Block 2 = 512 real tokens

So 512 is the fixed token length used to make training efficient.

Is 512 Embedding Size?
No.

Embedding size means the hidden vector dimension of the model.

For example, a model may convert each token into a vector like:

token → 2048-dimensional vector

That 2048 is embedding/hidden size.

But 512 here means:

How many tokens we give to the model at one time

In [34]:
# ============================================================
# 12. Tokenization and text packing
# ============================================================

# Appproach 1
def tokenize_function(examples):
    # Tokenize text without padding. Padding is handled dynamically by the collator.
    return tokenizer(examples["text"])

In [35]:
tokenized_datasets = raw_datasets.map(
    tokenize_function,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
    desc="Tokenizing text corpus",
)

tokenized_datasets

Tokenizing text corpus:   0%|          | 0/7 [00:00<?, ? examples/s]

Tokenizing text corpus:   0%|          | 0/2 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 7
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 2
    })
})

| Parameter                                       | Meaning                                                                                                                               |
| ----------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------- |
| `tokenize_function`                             | This function converts each text example into token IDs.                                                                              |
| `batched=True`                                  | The function processes multiple rows at once instead of one row at a time. This makes tokenization faster.                            |
| `remove_columns=datasets["train"].column_names` | After tokenization, the original dataset columns are removed. Only tokenized columns such as `input_ids` and `attention_mask` remain. |
| `desc="Tokenizing text corpus"`                 | This message is shown in the progress bar so we can understand that tokenization is currently running.                                |


In [36]:
# Check sample
tokenized_datasets['train']['input_ids'][0]

[1,
 1963,
 22824,
 28460,
 26101,
 3630,
 448,
 9305,
 29871,
 29945,
 9305,
 29871,
 29945,
 448,
 319,
 29902,
 297,
 360,
 11124,
 8565,
 22205,
 322,
 1963,
 22824,
 346,
 329,
 936,
 390,
 29987,
 29928,
 29936,
 1963,
 22824,
 29899,
 7247,
 1034,
 13364,
 6081,
 363,
 2888,
 2691,
 29899,
 29873,
 27964,
 322,
 390,
 10051,
 7639,
 362,
 29889,
 7519,
 29883,
 1288,
 2793,
 871,
 29936,
 451,
 16083,
 9848,
 29889,
 17157,
 29769,
 3012,
 928,
 616,
 21082,
 338,
 10231,
 368,
 1304,
 297,
 1374,
 22824,
 346,
 329,
 936,
 5925,
 304,
 27599,
 20853,
 1199,
 29892,
 1301,
 924,
 290,
 1199,
 29892,
 3279,
 290,
 1199,
 29892,
 17135,
 17292,
 327,
 7384,
 29892,
 22233,
 9562,
 29892,
 322,
 24899,
 936,
 20035,
 29889,
 512,
 3646,
 29769,
 29892,
 4933,
 6509,
 4733,
 508,
 7536,
 277,
 675,
 2531,
 267,
 470,
 3279,
 1144,
 393,
 1122,
 1708,
 3269,
 284,
 16178,
 297,
 17135,
 4768,
 3002,
 29889,
 4525,
 27303,
 526,
 9324,
 6419,
 746,
 23387,
 411,
 17986,
 8845,
 29892,

```python
# def tokenize_function(examples):
#     # Tokenize text without padding. Padding is handled dynamically by the collator.
#     return tokenizer(examples["text"])

tokenizer(examples["text"])

tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=512
)

tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])

trainer = Trainer(
     model=model,
     args=training_args,
     train_dataset=tokenized
)
```

# Approach 1: Without Text Packing

Each paragraph/example ko directly 512 token length me convert kar diya:

- Paragraph 1 → tokenize → pad/truncate to 512 tokens
- Paragraph 2 → tokenize → pad/truncate to 512 tokens
- Paragraph 3 → tokenize → pad/truncate to 512 tokens

## Example

```text
Paragraph has 100 real tokens

Padding added = 412 tokens

Final length = 512 tokens
```

## Good for

- Beginner teaching
- Small demo
- Simple notebook
- Less complex explanation

## Problem

- Lots of padding
- GPU wastage
- Less efficient training

---

# Approach 2: With Text Packing

All tokenized text ko join karke fixed blocks banata hai:

```text
Paragraph 1 tokens + Paragraph 2 tokens + Paragraph 3 tokens
↓
One long token stream
↓
Split into 512-token blocks
```

## Example

```text
Paragraph 1 = 100 tokens

Paragraph 2 = 150 tokens

Paragraph 3 = 262 tokens

Together = 512 real tokens
```

Yaha padding waste nahi hota.

## Good for

- Better GPU utilization
- More efficient continued pretraining
- More real tokens per batch
- Industry-style causal LM pretraining

In [37]:
# Approach 2:
def create_training_blocks(tokenized_examples):
    # Join all token IDs from multiple examples into one long list.
    all_input_ids = []
    all_attention_masks = []

    for input_ids in tokenized_examples["input_ids"]:
        all_input_ids.extend(input_ids)

    for attention_mask in tokenized_examples["attention_mask"]:
        all_attention_masks.extend(attention_mask)

    # Calculate how many complete blocks we can create.
    total_tokens = len(all_input_ids)
    usable_tokens = (total_tokens // config.block_size) * config.block_size

    # If we do not have enough tokens to create even one block, return empty data.
    if usable_tokens == 0:
        return {
            "input_ids": [],
            "attention_mask": [],
            "labels": [],
        }

    # Keep only tokens that can fit into complete fixed-size blocks.
    all_input_ids = all_input_ids[:usable_tokens]
    all_attention_masks = all_attention_masks[:usable_tokens]

    # Split the long token list into fixed-size training blocks.
    input_id_blocks = []
    attention_mask_blocks = []

    for start_index in range(0, usable_tokens, config.block_size):
        end_index = start_index + config.block_size

        input_id_blocks.append(all_input_ids[start_index:end_index])
        attention_mask_blocks.append(all_attention_masks[start_index:end_index])

    # For causal language modeling, labels are the same as input IDs.
    # The model uses these labels to learn next-token prediction.
    labels = input_id_blocks.copy()

    return {
        "input_ids": input_id_blocks,
        "attention_mask": attention_mask_blocks,
        "labels": labels,
    }

## What Does This Function Do?

This function converts tokenized text into **fixed-size training blocks**.

First, it joins all token IDs into one long sequence.  
Then, it cuts that long sequence into equal blocks of `config.block_size` tokens.

For causal language modeling, the labels are copied from `input_ids` because the model learns to predict the next token.

---

## Example

Suppose we have these tokenized inputs:

```text
Input token lists:

[10, 20, 30]
[40, 50]
[60, 70, 80, 90]
```

After joining all token lists together:

```text
[10, 20, 30, 40, 50, 60, 70, 80, 90]
```

If:

```python
block_size = 4
```

Then the final training blocks become:

```text
Block 1 = [10, 20, 30, 40]
Block 2 = [50, 60, 70, 80]
```

The remaining token is:

```text
[90]
```

This token is dropped because it cannot form a complete block of 4 tokens.

---

### Simple Summary

This step prepares the final causal language modeling dataset by converting many small tokenized examples into equal-length token blocks.

In [38]:
final_dataset = tokenized_datasets.map(
    create_training_blocks,
    batched=True,
    desc=f"Creating fixed-size training blocks of {config.block_size} tokens",
)

Creating fixed-size training blocks of 512 tokens:   0%|          | 0/7 [00:00<?, ? examples/s]

Creating fixed-size training blocks of 512 tokens:   0%|          | 0/2 [00:00<?, ? examples/s]

In [39]:
sample = final_dataset["train"][0]

In [40]:
print("Keys:", sample.keys())
print("input_ids length:", len(sample["input_ids"]))
print("labels length:", len(sample["labels"]))
print("Decoded sample preview:\n")
print(tokenizer.decode(sample["input_ids"][:250]))

Keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
input_ids length: 512
labels length: 512
Decoded sample preview:

<s> Pharma Domain Training Data - Page 5 Page 5 - AI in Drug Discovery and Pharmaceutical R&D; Pharma-domain corpus extension for custom fine-tuning and RAG experimentation. Educational content only; not medical advice. Target identification Artificial intelligence is increasingly used in pharmaceutical research to analyze genomics, transcriptomics, proteomics, disease phenotypes, chemical libraries, and clinical datasets. In target identification, machine learning models can prioritize genes or proteins that may play causal roles in disease biology. These predictions are strengthened when integrated with experimental validation, pathway analysis, human genetics, and disease-relevant biomarkers. Molecular screening In early discovery, deep learning can support virtual screening by predicting protein-ligand binding affinity, molecular properties, toxicity signals,

### 13. Load model with QLoRA-friendly configuration

- We use 4-bit quantized loading when CUDA is available.

Why?
- Lower GPU memory usage
- Faster experimentation
- Practical for Colab-style environments
- Common industry approach for parameter-efficient fine-tuning

If CUDA is not available, the notebook falls back to normal loading, but training will be slow on CPU.

In [41]:
# ============================================================
# 13. Load base model
# ============================================================

use_cuda = torch.cuda.is_available()
print("CUDA available:", use_cuda)
if use_cuda:
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [42]:
# Clear memory before loading the model.
gc.collect()
if use_cuda:
    torch.cuda.empty_cache()

In [43]:
if use_cuda:
    from transformers import BitsAndBytesConfig
    from peft import prepare_model_for_kbit_training

    # Configure 4-bit quantization to reduce GPU memory usage.
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    # Load the base model in 4-bit mode on available GPU devices.
    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        quantization_config=quantization_config,
        device_map="auto",
        trust_remote_code=True,
    )

    # Prepare the quantized model for stable LoRA/QLoRA training.
    base_model = prepare_model_for_kbit_training(base_model)

else:
    # Load the base model normally when GPU is not available.
    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

# Disable cache during training to reduce memory usage and avoid training warnings.
base_model.config.use_cache = False

print("Base model loaded successfully.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Base model loaded successfully.


### Apply LoRA adapters

In [44]:
# ============================================================
# 14. Apply LoRA adapters
# ============================================================
# LoRA trains a small number of adapter parameters instead of updating all base model weights.
# This is cheaper than full fine-tuning and is widely used in real projects.

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=config.lora_r,
    lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)


In [45]:
# Final LoRA enabled Model
model = get_peft_model(base_model, lora_config)

In [46]:
# Check number of trainable parameters
model.print_trainable_parameters()

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


### Data Collator

In [47]:
# ============================================================
# 15. Data collator
# ============================================================

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

## Why Do We Need `DataCollator For LanguageModeling`?

After tokenization and text packing, our dataset contains token IDs in a training-ready structure.

However, the `Trainer` still needs a component that can take multiple examples from the dataset and convert them into a proper batch during training.

That component is called a **data collator**.

```python
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)
```

---

## What Does the Data Collator Do?

The data collator prepares mini-batches for the model.

It handles things like:

- Collecting multiple training examples together
- Padding sequences if required
- Converting examples into tensors
- Preparing labels for language modeling

---

## Example

Suppose our packed dataset has training examples like this:

```text
Example 1 = 512 tokens
Example 2 = 512 tokens
Example 3 = 512 tokens
```

During training, the Trainer may take two examples at a time:

```text
Batch = Example 1 + Example 2
```

The data collator converts them into tensors like:

```text
input_ids shape      = [2, 512]
attention_mask shape = [2, 512]
labels shape         = [2, 512]
```

This is the format the model expects during training.

---

## Why `mlm=False`?

`mlm` means **Masked Language Modeling**.

Masked Language Modeling is used for BERT-style models.

### Example

```text
Metformin is used for [MASK].
```

The model predicts the masked word:

```text
diabetes
```

But we are using TinyLlama, which is a causal language model.

Causal language models learn by predicting the next token from left to right.

### Example

```text
Metformin → is
Metformin is → used
Metformin is used → for
Metformin is used for → diabetes
```

So we set:

```python
mlm=False
```

This tells Hugging Face:

> Do not use BERT-style masked language modeling. Use causal language modeling instead.

---

## Why Is This Needed Even After Tokenization and Packing?

- Tokenization converts text into token IDs.
- Text packing groups token IDs into fixed-size blocks.
- But the data collator prepares those blocks into actual training batches.

---

## Complete Training Pipeline

```text
Raw pharma text
   ↓
Tokenization
   ↓
Token IDs
   ↓
Text packing
   ↓
Fixed-size training blocks
   ↓
Data collator
   ↓
Mini-batches for Trainer
   ↓
Model training
```

### Training Arguments

In [48]:
# ============================================================
# 16. Training arguments
# ============================================================
# These settings are designed for a small classroom/demo run.
# For larger corpora, increase dataset size, epochs, and evaluation frequency carefully.

from transformers import TrainingArguments

In [49]:
training_kwargs = dict(
    output_dir=config.output_dir,
    num_train_epochs=config.num_train_epochs,
    max_steps=config.max_steps,
    per_device_train_batch_size=config.per_device_train_batch_size,
    per_device_eval_batch_size=config.per_device_eval_batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,
    warmup_steps=5,
    weight_decay=config.weight_decay,

    # Log training loss at every step for small demo datasets.
    logging_steps=1,
    logging_first_step=True,

    eval_steps=config.eval_steps,
    save_steps=config.save_steps,
    save_total_limit=config.save_total_limit,
    fp16=use_cuda,
    bf16=False,
    report_to="none",
    remove_unused_columns=False,
)

In [50]:
training_args = TrainingArguments(**training_kwargs)

### Initialize Trainer

In [51]:
# ============================================================
# 17. Build Trainer
# ============================================================
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=final_dataset["train"],
    eval_dataset=final_dataset["validation"],
    data_collator=data_collator,
)
print("Trainer is ready.")

Trainer is ready.


### Start Training

In [52]:
# ============================================================
# 18. Start training
# ============================================================
train_result = trainer.train()

print("Training completed.")

Step,Training Loss
1,2.148971
2,2.148971
3,2.124051
4,2.073108
5,1.995582
6,1.889092
7,1.752804
8,1.644385
9,1.564879
10,1.505728


Training completed.


In [53]:
# check logs
for log in trainer.state.log_history:
    print(log)

{'loss': 2.14897084236145, 'grad_norm': 0.6806048154830933, 'learning_rate': 0.0, 'epoch': 1.0, 'step': 1}
{'loss': 2.1489710807800293, 'grad_norm': 0.6922855973243713, 'learning_rate': 4e-05, 'epoch': 2.0, 'step': 2}
{'loss': 2.1240506172180176, 'grad_norm': 0.6668626666069031, 'learning_rate': 8e-05, 'epoch': 3.0, 'step': 3}
{'loss': 2.073108196258545, 'grad_norm': 0.6350870728492737, 'learning_rate': 0.00012, 'epoch': 4.0, 'step': 4}
{'loss': 1.995582103729248, 'grad_norm': 0.6457183361053467, 'learning_rate': 0.00016, 'epoch': 5.0, 'step': 5}
{'loss': 1.8890916109085083, 'grad_norm': 0.681577742099762, 'learning_rate': 0.0002, 'epoch': 6.0, 'step': 6}
{'loss': 1.7528035640716553, 'grad_norm': 0.6949010491371155, 'learning_rate': 0.00016, 'epoch': 7.0, 'step': 7}
{'loss': 1.6443846225738525, 'grad_norm': 0.6966458559036255, 'learning_rate': 0.00012, 'epoch': 8.0, 'step': 8}
{'loss': 1.5648794174194336, 'grad_norm': 0.7203812599182129, 'learning_rate': 8e-05, 'epoch': 9.0, 'step': 9}

## Save adapter and tokenizer

In [54]:
# ============================================================
# 19. Save adapter and tokenizer
# ============================================================

trainer.model.save_pretrained(config.adapter_dir)
tokenizer.save_pretrained(config.adapter_dir)

print(f"LoRA adapter saved to: {config.adapter_dir}")
print("Saved files:")
print(os.listdir(config.adapter_dir))

LoRA adapter saved to: /content/pharma_tinyllama_lora_adapter
Saved files:
['adapter_model.safetensors', 'tokenizer_config.json', 'adapter_config.json', 'README.md', 'tokenizer.json']


### Push LoRA adapter to Hugging Face Hub

In [55]:
from huggingface_hub import login
from google.colab import userdata

# Retrieve the token from Colab's secrets manager
HF_TOKEN = userdata.get("HF_TOKEN_WRITE")

# Log in to Hugging Face Hub
login(token=HF_TOKEN)

print("Successfully logged in to Hugging Face Hub.")

Successfully logged in to Hugging Face Hub.


In [55]:
# ============================================================
# Push Stage 1 non-instruction LoRA adapter to Hugging Face
# ============================================================

trainer.model.push_to_hub(
    HF_REPO_NON_INSTRUCTION_ADAPTER,
    private=False
)


tokenizer.push_to_hub(
    HF_REPO_NON_INSTRUCTION_ADAPTER,
    private=False
)

print("Stage 1 non-instruction LoRA adapter pushed to:")
print(HF_REPO_NON_INSTRUCTION_ADAPTER)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 34.5kB / 50.5MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Stage 1 non-instruction LoRA adapter pushed to:
AreebAhmxd/pharma-tinyllama-1.1b-non-instruction-lora-adapter


### Reload base model + LoRA adapter correctly

In [56]:
# ============================================================
# 21. Reload base model + LoRA adapter correctly
# ============================================================

# Clean old objects to free memory.
del trainer
try:
    del model
    del base_model
except NameError:
    pass

gc.collect()
if use_cuda:
    torch.cuda.empty_cache()

In [57]:
# Initialize the saved tokenizer
inference_tokenizer = AutoTokenizer.from_pretrained(config.adapter_dir, use_fast=True)

if inference_tokenizer.pad_token is None:
    inference_tokenizer.pad_token = inference_tokenizer.eos_token

In [58]:
# Initialize the base model
if use_cuda:
    inference_base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map="auto",
        trust_remote_code=True,
    )
else:
    inference_base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [59]:
# Add LoRA adapter on top of the base model
inference_model = PeftModel.from_pretrained(inference_base_model, config.adapter_dir)

In [60]:
# set model to evaluation mode
inference_model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [61]:
print("Base model + LoRA adapter loaded successfully for inference.")

Base model + LoRA adapter loaded successfully for inference.


### Inference Helper

In [62]:
# ============================================================
# 22. Inference helper
# ============================================================
# Since this is non-instruction fine-tuning, prompts should look like text continuations,
# not chat-style questions.

def generate_completion(prompt: str, max_new_tokens: int = 120) -> str:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    inputs = inference_tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=inference_tokenizer.eos_token_id,
            eos_token_id=inference_tokenizer.eos_token_id,
        )

    return inference_tokenizer.decode(outputs[0], skip_special_tokens=True)

### Test text continuation

In [63]:
# ============================================================
# 23. Test text continuation
# ============================================================
# These prompts are continuation-style prompts.
# In Stage 2, we will create instruction prompts for Q&A.

prompts = [
    "Metformin is one of the most widely prescribed oral antihyperglycemic agents",
    "Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe",
    "Artificial intelligence is transforming pharmaceutical research by accelerating",
]

In [ ]:
# generate output
for prompt in prompts:
    print("=" * 100)
    print("PROMPT:")
    print(prompt)
    print("\nMODEL CONTINUATION:")
    print(generate_completion(prompt, max_new_tokens=120))
    print()

[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT:
Metformin is one of the most widely prescribed oral antihyperglycemic agents

MODEL CONTINUATION:


[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Metformin is one of the most widely prescribed oral antihyperglycemic agents used in the treatment of type 2 diabetes mellitus. In a recent study, we aimed to evaluate the impact of metformin on adiposity and liver-related risk factors in patients with type 2 diabetes mellitus.
Patients included were 74 subjects (41 males) aged between 35 and 60 years with T2DM (n=49), T1DM (n=21), and prediabetic status (n=24). Body composition was assessed by dual

PROMPT:
Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe

MODEL CONTINUATION:


[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe, a drug that blocks cholesterol absorption in the intestine, results in greater reductions in LDL-C compared to monotherapy. The data support the use of Atorvastatin 80 mg with Ezetimibe 10 mg once daily for primary prevention of cardiovascular disease and diabetes mellitus in patients at increased risk of cardiovascular disease or diabetes mellitus. In addition, a clinical trial has shown that combining Atorvastatin 80

PROMPT:
Artificial intelligence is transforming pharmaceutical research by accelerating

MODEL CONTINUATION:
Artificial intelligence is transforming pharmaceutical research by accelerating drug discovery and improving clinical development, but it also creates new challenges. AI can help scientists predict how a compound might work in the body and understand its safety profile. It can also identify biomarkers that may predict disease progression and help inform clinical trial design. AI can hel

### Merge LoRa adapter to Base Model (Optional step)

In [64]:
# ============================================================
# 24. Optional merge step
# ============================================================
# This step merges the LoRA adapter into the base model.
# Use this only when you want a standalone model for deployment.

import os
import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel

merged_model_dir = "/content/pharma_tinyllama_merged_model"
os.makedirs(merged_model_dir, exist_ok=True)

In [65]:
# Reload the base model in float16 for safe merging.
base_model = AutoModelForCausalLM.from_pretrained(
    config.model_name,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True,
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [66]:
import torchao
print(torchao.__version__)

0.17.0


In [67]:
# Load the trained LoRA adapter on top of the base model.
model_with_adapter = PeftModel.from_pretrained(
    base_model,
    config.adapter_dir
)

In [68]:
# Merge LoRA adapter weights into the base model weights.
merged_model = model_with_adapter.merge_and_unload()

In [69]:
# Save the merged standalone model and tokenizer.

merged_model.save_pretrained(merged_model_dir)

inference_tokenizer.save_pretrained(merged_model_dir)

print(f"Merged model saved to: {merged_model_dir}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved to: /content/pharma_tinyllama_merged_model


In [70]:
# ============================================================
# 25. Push merged Stage 1 model to Hugging Face Hub
# ============================================================

# Defining a sensible repository name
repo_id = HF_REPO_NON_INSTRUCTION_MERGED

print(f"Pushing merged model to: {repo_id}...")

# Push the model weights
merged_model.push_to_hub(
    repo_id,
    private=False,
    commit_message="Initial push of merged pharma domain-adapted TinyLlama"
)

# Push the tokenizer
inference_tokenizer.push_to_hub(
    repo_id,
    private=False,
    commit_message="Add tokenizer for pharma domain-adapted TinyLlama"
)

print(f"Successfully pushed to: https://huggingface.co/{repo_id}")

Pushing merged model to: AreebAhmxd/pharma-tinyllama-1.1b-non-instruction-model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...xpfesm0/model.safetensors:   1%|          | 16.0MB / 2.20GB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Successfully pushed to: https://huggingface.co/AreebAhmxd/pharma-tinyllama-1.1b-non-instruction-model


## Continue with Instruction Fine-Tuning on the Same Domain-Adapted Finetuned Model
# Stage 2: Instruction Fine-Tuning

In Stage 1, we performed **non-instruction fine-tuning / domain-adaptive continued pretraining** on raw pharma PDF text.

Now we continue from the **same Stage 1 LoRA adapter** and perform **instruction fine-tuning** using structured pharma instruction-response examples.

```text
Base TinyLlama
   ↓
Stage 1: Raw pharma text continued pretraining using LoRA
   ↓
Stage 1 domain-adapted LoRA adapter
   ↓
Stage 2: Instruction fine-tuning on pharma Q&A data
   ↓
Final instruction-tuned pharma LoRA adapter
```

This means we are not starting from scratch. We are continuing from the model adapter trained in the previous stage.

---

## What Changes in Instruction Fine-Tuning?

For non-instruction fine-tuning, the data looked like raw text:

```text
Metformin is one of the most widely prescribed oral antihyperglycemic agents...
```

For instruction fine-tuning, the data looks like:

```json
{
  "instruction": "Explain the mechanism of action of Metformin.",
  "input": "",
  "output": "Metformin primarily activates AMPK..."
}
```

This teaches the model not only pharma language, but also how to answer user instructions.

### Load the Instruction Dataset

In [71]:
instruction_data_path = "/content/pharma_instruction_dataset.jsonl"

In [72]:
from datasets import load_dataset

In [73]:
instruction_dataset = load_dataset(
    "json",
    data_files=instruction_data_path,
    split="train"
)

Generating train split: 0 examples [00:00, ? examples/s]

In [74]:
print(instruction_dataset)

Dataset({
    features: ['instruction', 'input', 'output', 'source_page', 'topic'],
    num_rows: 48
})


In [75]:
print(instruction_dataset[0])

{'instruction': 'Explain the primary mechanism of action of metformin.', 'input': '', 'output': 'Metformin primarily acts by activating AMP-activated protein kinase, also called AMPK. AMPK is a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while reducing hepatic gluconeogenesis, which helps lower blood glucose levels.', 'source_page': 1, 'topic': 'Metformin pharmacology'}


### Format Instruction Dataset

In [76]:
# ============================================================
# Format instruction records
# ============================================================
# We convert every record into Alpaca-style training text.

def format_instruction_record(record):
    instruction = str(record.get("instruction", "")).strip()
    input_text = str(record.get("input", "")).strip()
    output_text = str(record.get("output", "")).strip()

    if input_text:
        text = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input_text}\n\n"
            f"### Response:\n{output_text}"
        )
    else:
        text = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Response:\n{output_text}"
        )

    return {"text": text}

In [77]:
instruction_dataset = instruction_dataset.map(format_instruction_record)

Map:   0%|          | 0/48 [00:00<?, ? examples/s]

In [78]:
instruction_dataset

Dataset({
    features: ['instruction', 'input', 'output', 'source_page', 'topic', 'text'],
    num_rows: 48
})

In [79]:
print(instruction_dataset[0]["text"])

### Instruction:
Explain the primary mechanism of action of metformin.

### Response:
Metformin primarily acts by activating AMP-activated protein kinase, also called AMPK. AMPK is a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while reducing hepatic gluconeogenesis, which helps lower blood glucose levels.


In [80]:
# ============================================================
# Create train-validation split
# ============================================================

instruction_datasets = instruction_dataset.train_test_split(
    test_size=0.15,
    seed=42
)

instruction_datasets["validation"] = instruction_datasets.pop("test")

print(instruction_datasets)
print("Train examples:", len(instruction_datasets["train"]))
print("Validation examples:", len(instruction_datasets["validation"]))

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'source_page', 'topic', 'text'],
        num_rows: 40
    })
    validation: Dataset({
        features: ['instruction', 'input', 'output', 'source_page', 'topic', 'text'],
        num_rows: 8
    })
})
Train examples: 40
Validation examples: 8


### Tokenization

In [81]:
# ============================================================
# Tokenize instruction dataset
# ============================================================
# The tokenizer converts text into token IDs for model training.

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(tokenizer.pad_token)

</s>


In [82]:
instruction_max_length = 512

In [83]:
def tokenize_instruction_function(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )

    # For causal LM, labels are copied from input_ids.
    tokens["labels"] = tokens["input_ids"].copy()

    # Ignore padding tokens in the loss calculation.
    tokens["labels"] = [
        [
            token if mask == 1 else -100
            for token, mask in zip(input_ids, attention_mask)
        ]
        for input_ids, attention_mask in zip(tokens["input_ids"], tokens["attention_mask"])
    ]

    return tokens

## Why Do We Use `-100` in Labels?

When we tokenize instruction data, all examples are not the same length.

### Example

```text
Example 1 = 20 tokens
Example 2 = 80 tokens
Example 3 = 150 tokens
```

But for training, we often make every example the same length, like:

```python
max_length = 512
```

So shorter examples get extra padding tokens.

### Example

```text
Real text tokens + padding tokens = 512 tokens
```

Now the problem is:

We want the model to learn from real text, not from padding.

So we use `-100` in labels.

`-100` tells PyTorch:

> Ignore this position while calculating loss.

In [84]:
instruction_tokenized_datasets = instruction_datasets.map(
    tokenize_instruction_function,
    batched=True,
    remove_columns=instruction_datasets["train"].column_names,
    desc="Tokenizing instruction dataset",
)

print(instruction_tokenized_datasets)

Tokenizing instruction dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Tokenizing instruction dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 40
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 8
    })
})


## We can also add LoRA adapter to initial Base model

In [ ]:
# # ============================================================
# # Reload Stage 1 LoRA adapter as trainable
# # ============================================================
# # We continue instruction fine-tuning from the non-instruction LoRA adapter.

# gc.collect()

# if torch.cuda.is_available():
#     torch.cuda.empty_cache()

# use_cuda = torch.cuda.is_available()

# if use_cuda:
#     instruction_base_model = AutoModelForCausalLM.from_pretrained(
#         config.model_name,
#         quantization_config=BitsAndBytesConfig(
#             load_in_4bit=True,
#             bnb_4bit_quant_type="nf4",
#             bnb_4bit_compute_dtype=torch.float16,
#             bnb_4bit_use_double_quant=True,
#         ),
#         device_map="auto",
#         trust_remote_code=True,
#     )

#     instruction_base_model = prepare_model_for_kbit_training(instruction_base_model)

# else:
#     instruction_base_model = AutoModelForCausalLM.from_pretrained(
#         config.model_name,
#         torch_dtype=torch.float32,
#         trust_remote_code=True,
#     )

# instruction_base_model.config.use_cache = False

# # Load the Stage 1 adapter and keep it trainable for Stage 2.
# instruction_model = PeftModel.from_pretrained(
#     instruction_base_model,
#     config.adapter_dir,
#     is_trainable=True,
# )

# instruction_model.print_trainable_parameters()

# Base model
#    +
# Stage 1 domain LoRA adapter
#    ↓ continue training
# Stage 1 + Stage 2 final LoRA adapter

| Point                          | Approach 1: Continue Same Stage 1 LoRA Adapter                                         | Approach 2: Merge Stage 1, Then Add New LoRA Adapter                                                      |
| ------------------------------ | -------------------------------------------------------------------------------------- | --------------------------------------------------------------------------------------------------------- |
| Flow                           | Base model + Stage 1 LoRA adapter → continue training same adapter on instruction data | Base model + Stage 1 LoRA adapter → merge → load merged model → add new LoRA adapter for instruction data |
| Main idea                      | The same adapter learns both domain language and instruction-following behavior        | Stage 1 knowledge becomes part of the merged model, then a new adapter learns instruction behavior        |
| Simplicity                     | Easier for students to understand                                                      | More complex because merge step is involved                                                               |
| Merge required before Stage 2? | No                                                                                     | Yes                                                                                                       |
| Risk/complexity                | Lower complexity                                                                       | Higher complexity, especially if Stage 1 model was loaded in 4-bit/QLoRA mode                             |
| Output size                    | Small final LoRA adapter                                                               | Large merged Stage 1 model + small Stage 2 adapter                                                        |
| Hugging Face upload            | Easy, only adapter can be pushed                                                       | Heavier because merged model is large                                                                     |
| Best for course/demo           | Best choice                                                                            | Use only if you already merged Stage 1                                                                    |
| Best for production            | Good for experimentation and adapter-based deployment                                  | Useful when you want Stage 1 knowledge permanently inside the base model                                  |
| Recommended for your notebook? | **Yes, recommended**                                                                   | Only if Stage 1 adapter is already merged                                                                 |


## Load merged Stage 1 model and add new LoRA adapter for instruction tuning
---



In [85]:
# ============================================================
# Load merged Stage 1 model and add new LoRA adapter for instruction tuning
# ============================================================

# Merged Stage 1 Model
#    +
# New LoRA adapter for instruction tuning

import gc
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

use_cuda = torch.cuda.is_available()

merged_model_dir = "/content/pharma_tinyllama_merged_model"

if use_cuda:
    # Load merged Stage 1 model in 4-bit mode for QLoRA instruction tuning.
    instruction_base_model = AutoModelForCausalLM.from_pretrained(
        merged_model_dir,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map="auto",
        trust_remote_code=True,
    )

    instruction_base_model = prepare_model_for_kbit_training(instruction_base_model)

else:
    # CPU fallback. Training on CPU will be slow.
    instruction_base_model = AutoModelForCausalLM.from_pretrained(
        merged_model_dir,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

instruction_base_model.config.use_cache = False

# Create a new LoRA adapter for instruction fine-tuning.
instruction_lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

instruction_model = get_peft_model(
    instruction_base_model,
    instruction_lora_config
)

instruction_model.print_trainable_parameters()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


## Instruction fine-tuning data collator

In [86]:
# ============================================================
# Instruction fine-tuning data collator
# ============================================================
# This prepares mini-batches for causal language model training.

from transformers import DataCollatorForLanguageModeling
instruction_data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

In [87]:
# ============================================================
# Instruction fine-tuning arguments
# ============================================================

instruction_output_dir = "/content/pharma_tinyllama_instruction_lora_output"
instruction_adapter_dir = "/content/pharma_tinyllama_instruction_lora_adapter"

os.makedirs(instruction_output_dir, exist_ok=True)
os.makedirs(instruction_adapter_dir, exist_ok=True)

In [88]:
from transformers import TrainingArguments

instruction_training_args = TrainingArguments(
    output_dir=instruction_output_dir,

    # Train for 5 full epochs.
    num_train_epochs=5,
    max_steps=-1,

    # Batch settings.
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    # Optimizer settings.
    learning_rate=1e-4,
    warmup_steps=5,
    weight_decay=0.01,

    # Show training loss at every step.
    logging_steps=1,
    logging_first_step=True,

    # Run validation at every step.
    eval_strategy="steps",
    eval_steps=1,

    # Save checkpoints.
    save_steps=25,
    save_total_limit=2,

    # Precision settings.
    fp16=use_cuda,
    bf16=False,

    # Disable external logging tools.
    report_to="none",

    # Keep required columns.
    remove_unused_columns=False,
)

print(instruction_training_args)

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=1,
eval_strategy=IntervalStrategy.STEPS,
eval_us

## Build instruction Trainer

In [89]:
# ============================================================
# Build instruction Trainer
# ============================================================

from transformers import Trainer
instruction_trainer = Trainer(
    model=instruction_model,
    args=instruction_training_args,
    train_dataset=instruction_tokenized_datasets["train"],
    eval_dataset=instruction_tokenized_datasets["validation"],
    data_collator=instruction_data_collator,
)

print("Instruction Trainer is ready.")

Instruction Trainer is ready.


## Training instruction fine-tuned model

In [90]:
# ============================================================
# Start instruction fine-tuning
# ============================================================

instruction_train_result = instruction_trainer.train()


Step,Training Loss,Validation Loss
1,1.810061,2.114269
2,2.014070,2.095296
3,2.296757,2.055144
4,1.947558,2.000345
5,1.996108,1.934622
6,1.828667,1.862439
7,1.605513,1.795707
8,1.588237,1.731977
9,1.703012,1.673789
10,1.709921,1.628267


In [91]:
print("Instruction fine-tuning completed.")
print(instruction_train_result)

Instruction fine-tuning completed.
TrainOutput(global_step=25, training_loss=1.5227490758895874, metrics={'train_runtime': 129.1935, 'train_samples_per_second': 1.548, 'train_steps_per_second': 0.194, 'total_flos': 643355482521600.0, 'train_loss': 1.5227490758895874, 'epoch': 5.0})


## Save final instruction-tuned LoRA adapter

In [92]:
# ============================================================
# Save final instruction-tuned LoRA adapter
# ============================================================
# This adapter now contains Stage 1 domain adaptation + Stage 2 instruction tuning.

import os

instruction_adapter_dir = "/content/pharma_tinyllama_instruction_lora_adapter"
os.makedirs(instruction_adapter_dir, exist_ok=True)

instruction_trainer.model.save_pretrained(instruction_adapter_dir)
tokenizer.save_pretrained(instruction_adapter_dir)

print(f"Final instruction-tuned LoRA adapter saved to: {instruction_adapter_dir}")
print(os.listdir(instruction_adapter_dir))

Final instruction-tuned LoRA adapter saved to: /content/pharma_tinyllama_instruction_lora_adapter
['adapter_model.safetensors', 'tokenizer_config.json', 'adapter_config.json', 'README.md', 'tokenizer.json']


## Reload final instruction-tuned adapter for inference

In [99]:
# ============================================================
# Reload final instruction-tuned adapter for inference
# ============================================================

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

if use_cuda:
    base_model = AutoModelForCausalLM.from_pretrained(
        merged_model_dir,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map="auto",
        trust_remote_code=True,
    )
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

final_instruction_model = PeftModel.from_pretrained(
    base_model,
    instruction_adapter_dir,
)

final_instruction_model.eval()

print("Final instruction-tuned model loaded successfully.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Final instruction-tuned model loaded successfully.


## Push Instruction FT LoRA Adapter to Hugging Face

In [94]:
# ============================================================
# 26. Push Instruction-Tuned LoRA adapter to Hugging Face Hub
# ============================================================

# Defining a sensible repository name for the Stage 2 adapter
instruction_repo_id = HF_REPO_INSTRUCTION_ADAPTER

print(f"Pushing instruction-tuned adapter to: {instruction_repo_id}...")

# Push the adapter weights
instruction_model.push_to_hub(
    instruction_repo_id,
    private=False,
    commit_message="Pushing Stage 2 pharma instruction-tuned LoRA adapter"
)

# Push the tokenizer
tokenizer.push_to_hub(
    instruction_repo_id,
    private=False,
    commit_message="Add tokenizer for pharma instruction-tuned model"
)

print(f"Successfully pushed to: https://huggingface.co/{instruction_repo_id}")

Pushing instruction-tuned adapter to: AreebAhmxd/pharma-tinyllama-1.1b-instruction-lora-adapter...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 34.5kB / 50.5MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Successfully pushed to: https://huggingface.co/AreebAhmxd/pharma-tinyllama-1.1b-instruction-lora-adapter


## Inferencing

In [95]:
# ============================================================
# Instruction-style inference helper
# ============================================================

def build_instruction_prompt(instruction, input_text=""):
    instruction = instruction.strip()
    input_text = input_text.strip()

    if input_text:
        return (
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input_text}\n\n"
            f"### Response:\n"
        )

    return (
        f"### Instruction:\n{instruction}\n\n"
        f"### Response:\n"
    )


def generate_instruction_response(instruction, input_text="", max_new_tokens=150):
    prompt = build_instruction_prompt(instruction, input_text)

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(final_instruction_model.device)

    with torch.no_grad():
        outputs = final_instruction_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [96]:
# ============================================================
# Test instruction-tuned pharma model
# ============================================================

test_questions = [
    "Explain the primary mechanism of action of metformin.",
    "Why can atorvastatin and ezetimibe reduce LDL-C more effectively together?",
    "Summarize the role of lipid nanoparticles in mRNA vaccines.",
    "Why should AI predictions in drug discovery be experimentally validated?",
]

for question in test_questions:
    print("=" * 100)
    print("QUESTION:")
    print(question)

    print("\nMODEL RESPONSE:")
    print(generate_instruction_response(question, max_new_tokens=150))

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION:
Explain the primary mechanism of action of metformin.

MODEL RESPONSE:


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Explain the primary mechanism of action of metformin.

### Response:
Metformin is a biguanide that inhibits glycogen synthase kinase-1 (GSK1) activity, which prevents glucose from being converted to fatty acids. This causes the reduction of hepatic insulin resistance and lipolysis. Metformin can be taken as an oral agent, parenteral formulation, or as a combination therapy with other antidiabetic agents. It is approved for use in patients with type 2 diabetes mellitus.

### Explain the mechanism of action of pioglitazone.
Pioglitazone is a sulfonylurea that has multiple actions on glycemic control
QUESTION:
Why can atorvastatin and ezetimibe reduce LDL-C more effectively together?

MODEL RESPONSE:


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Why can atorvastatin and ezetimibe reduce LDL-C more effectively together?

### Response:
Ezetimibe is a drug that lowers cholesterol in the blood by blocking an enzyme called the LDL receptor. Atorvastatin is a statin, which helps to lower triglycerides in the blood. Together these drugs work to lower cholesterol in the blood.

### Instruction:
How does simvastatin affect the metabolism of dietary fats?

### Response:
Simvastatin increases the amount of cholesterol that is absorbed from the intestine into the bloodstream. This increase in cholesterol causes triglyceride levels to increase.

### Instruction:
QUESTION:
Summarize the role of lipid nanoparticles in mRNA vaccines.

MODEL RESPONSE:


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Summarize the role of lipid nanoparticles in mRNA vaccines.

### Response:
Lipid nanoparticles can be used to stabilize the RNA and enhance its transfection. They can also be used to increase the amount of the mRNA protein for subsequent delivery to target cells. Lipid nanoparticles are less prone to degradation, which helps maintain stability of mRNA in cellular environments. 

### Interviewee:
<NAME>, PhD

### Date:
10/20/2021

### Topic:
Chemistry

### Instruction:
Describe the structure of a carboxylic acid.

### Response:
A carboxylic acid is an organ
QUESTION:
Why should AI predictions in drug discovery be experimentally validated?

MODEL RESPONSE:
### Instruction:
Why should AI predictions in drug discovery be experimentally validated?

### Response:
AI-guided prediction of drug targets, drug-like molecules, and ligands can help predict biological activities. AI-guided validation is necessary to validate the predictions from the AI-based prediction models. The e

## Merge instruction-tuned LoRA adapter by having Non-Instruction tuned model as base model

In [100]:
# ============================================================
# Merge instruction-tuned LoRA adapter into base model
# ============================================================
# This creates a standalone instruction-tuned model.
# Later, we can use this merged model as the base model for preference tuning.

import os
import gc
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Path where the final merged instruction-tuned model will be saved.
merged_instruction_model_dir = "/content/pharma_tinyllama_instruction_merged_model"

os.makedirs(merged_instruction_model_dir, exist_ok=True)

# Load the original base model in normal precision for safe merging.
base_model_for_merge = AutoModelForCausalLM.from_pretrained(
    merged_model_dir,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)

# Load the tokenizer.
tokenizer_for_merge = AutoTokenizer.from_pretrained(
    config.model_name,
    trust_remote_code=True,
)

if tokenizer_for_merge.pad_token is None:
    tokenizer_for_merge.pad_token = tokenizer_for_merge.eos_token

# Attach the final instruction-tuned LoRA adapter.
model_with_instruction_adapter = PeftModel.from_pretrained(
    base_model_for_merge,
    instruction_adapter_dir,
)

# Merge LoRA adapter weights into the base model weights.
merged_instruction_model = model_with_instruction_adapter.merge_and_unload()

# Save the standalone merged model and tokenizer.
merged_instruction_model.save_pretrained(merged_instruction_model_dir)
tokenizer_for_merge.save_pretrained(merged_instruction_model_dir)

print(f"Merged instruction-tuned model saved to: {merged_instruction_model_dir}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged instruction-tuned model saved to: /content/pharma_tinyllama_instruction_merged_model


## Push Instruction FineTuned Model to Hugging Face

In [101]:
# ============================================================
# 27. Push Final Instruction-Tuned Model to Hugging Face Hub
# ============================================================

# Defining a more descriptive repository name for the final Stage 2 model
final_merged_repo_id = HF_REPO_INSTRUCTION_MERGED

print(f"Pushing final pharma assistant model to: {final_merged_repo_id}...")

# Push the standalone model weights
merged_instruction_model.push_to_hub(
    final_merged_repo_id,
    private=False,
    commit_message="Pushing final pharma-domain instruction-tuned assistant model"
)

# Push the tokenizer
tokenizer_for_merge.push_to_hub(
    final_merged_repo_id,
    private=False,
    commit_message="Add tokenizer for final pharma assistant model"
)

print(f"Successfully pushed to: https://huggingface.co/{final_merged_repo_id}")

Pushing final pharma assistant model to: AreebAhmxd/pharma-tinyllama-1.1b-instruction-model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...j42sulc/model.safetensors:   3%|3         | 71.9MB / 2.20GB            

Successfully pushed to: https://huggingface.co/AreebAhmxd/pharma-tinyllama-1.1b-instruction-model


## Fine-Tuning Stages Comparison

| Fine-tuning stage | Data format | What does the model learn? |
|------------------|------------|----------------------------|
| **Non-instruction FT** | Raw text | Domain language, terminology, and writing style |
| **Instruction FT** | Instruction → Response | How to answer user instructions and questions |
| **DPO Preference Tuning** | Prompt → Chosen vs Rejected | How to prefer better responses over weaker ones |

# Stage 3: Preference Tuning with DPO

In Stage 1, we adapted the model to the pharma domain using raw non-instruction text.

In Stage 2, we instruction-tuned the model using instruction-response data.

In Stage 3, we will use **preference data** with DPO.

DPO data has three main columns:

```text
prompt
chosen
rejected
```

- `prompt` is the user instruction.
- `chosen` is the preferred or better response.
- `rejected` is the less preferred or weaker response.

The goal of DPO is to make the model prefer the `chosen` response over the `rejected` response.

### Reference Paper

Paper link:

```text
https://arxiv.org/pdf/2305.18290
```

## DPO (Direct Preference Optimization) Summary

| Section | Simple Meaning | Key Point |
|----------|----------|----------|
| **DPO Full Form** | Direct Preference Optimization | The model is trained directly using preference data. |
| **Paper** | *Direct Preference Optimization* | Published at NeurIPS 2023 by Stanford researchers. |
| **Main Idea** | Teach the model which answer is better and which answer is weaker. | The model learns to prefer the `chosen` answer and avoid the `rejected` answer. |
| **Before DPO: RLHF** | RLHF usually has three stages. | SFT → Reward Model → PPO |
| **RLHF Problem** | RLHF is complex, expensive, and unstable. | Training a reward model and using PPO require high compute and careful tuning. |
| **DPO Insight** | A separate reward model is not required. | The language model itself can act like an implicit reward model. |
| **DPO Dataset Format** | Each sample has three main fields. | `prompt`, `chosen`, and `rejected` |
| **Prompt** | The user question or instruction. | Example: *"Explain the mechanism of metformin."* |
| **Chosen** | The better or preferred answer. | Usually accurate, complete, safe, and well-structured. |
| **Rejected** | The weaker or rejected answer. | Usually vague, incomplete, incorrect, or unsafe. |
| **DPO Training Goal** | Increase the probability of the preferred answer. | The model becomes more likely to generate answers similar to the `chosen` response. |
| **Role of Rejected Answer** | Shows the model what type of answer to avoid. | The model reduces the probability of the `rejected` style answer. |
| **Reference Model** | Usually the SFT model. | It prevents the DPO model from drifting too far from the original fine-tuned model. |
| **Policy Model** | The model being trained during DPO. | It learns to prefer the `chosen` answer over the `rejected` answer. |
| **Beta (β)** | A control parameter. | It controls how strongly the model moves away from the reference model. |
| **DPO Loss** | A binary classification-style loss. | It trains the model to make the `chosen` answer win over the `rejected` answer. |
| **Reward Model Needed?** | No. | DPO removes the need for separate reward model training. |
| **PPO Needed?** | No. | DPO works more like supervised training instead of reinforcement learning. |
| **Sampling During Training?** | No. | DPO does not require an expensive generation loop like PPO. |
| **Main Advantage** | Simpler and more stable. | Easier to implement compared to traditional RLHF. |
| **Training Cost** | Lower than RLHF. | Only the policy model is trained. |
| **DPO vs SFT** | SFT teaches the model how to answer. | DPO teaches the model which answer is better. |
| **DPO vs RLHF** | RLHF uses a reward model and PPO. | DPO directly uses preference loss. |
| **Gradient Intuition** | Stronger updates happen when the model ranks answers incorrectly. | The model learns more from difficult examples. |
| **Practical Pipeline** | Start with an SFT model, add preference data, then train with DPO. | This creates a simple alignment pipeline. |
| **Common Beta Value** | The paper commonly used `β = 0.1`. | For summarization tasks, `β = 0.5` was also used. |
| **Experiments** | Tested on sentiment, summarization, and dialogue tasks. | DPO can perform equal to or better than PPO. |
| **Limitation** | Large-scale training, reward hacking, and out-of-distribution generalization are still open questions. | DPO is powerful, but not perfect. |
| **Classroom One-Liner** | DPO teaches the model which answer is better. | Instruction tuning teaches answering; DPO teaches preference. |

In [102]:
# ============================================================
# 25. Install TRL for DPO training
# ============================================================
# TRL provides DPOTrainer and DPOConfig for preference tuning.

!pip install -q -U trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 12.7 MB/s eta 0:00:00


### Load the dataset

In [103]:
# ============================================================
# 26. Load DPO preference dataset
# ============================================================
# Expected columns: prompt, chosen, rejected

from datasets import load_dataset

preference_data_path = "/content/pharma_preference_dataset.jsonl"

preference_dataset = load_dataset(
    "json",
    data_files=preference_data_path,
    split="train"
)

print(preference_dataset)
print(preference_dataset[0])


Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['prompt', 'chosen', 'rejected', 'source_page', 'topic'],
    num_rows: 48
})
{'prompt': '### Instruction:\nExplain the primary mechanism of action of metformin.\n\n### Response:\n', 'chosen': 'Metformin primarily acts by activating AMP-activated protein kinase, also called AMPK. AMPK is a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while reducing hepatic gluconeogenesis, which helps lower blood glucose levels.', 'rejected': 'Metformin mainly works by increasing insulin secretion from the pancreas, and kidney function is usually not very relevant. Its side effects are generally not important unless the patient feels very sick.', 'source_page': 1, 'topic': 'Metformin pharmacology'}


### Train Validation split

In [104]:
# Create train-validation split
preference_dataset = preference_dataset.train_test_split(
    test_size=0.15,
    seed=42
)

# Rename test split to validation split
preference_dataset["validation"] = preference_dataset.pop("test")

print("After train-validation split:")
print(preference_dataset)
print("Train rows:", len(preference_dataset["train"]))
print("Validation rows:", len(preference_dataset["validation"]))

After train-validation split:
DatasetDict({
    train: Dataset({
        features: ['prompt', 'chosen', 'rejected', 'source_page', 'topic'],
        num_rows: 40
    })
    validation: Dataset({
        features: ['prompt', 'chosen', 'rejected', 'source_page', 'topic'],
        num_rows: 8
    })
})
Train rows: 40
Validation rows: 8


## Preference Tuning Base Model

For DPO, we use the **merged instruction-tuned model** as the base model.

Then we attach a **new LoRA adapter** for preference tuning.

This gives us the flow:

```text
Merged instruction-tuned model
        +
New preference LoRA adapter
        ↓
DPO preference tuning
```

## Load Merged Instruction model as base for preference tuning

In [105]:
merged_instruction_model_dir

'/content/pharma_tinyllama_instruction_merged_model'

In [106]:
# ============================================================
# Load merged instruction-tuned model as base for preference tuning
# ============================================================

from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

use_cuda = torch.cuda.is_available()

# The base model for preference tuning is the merged instruction-tuned model from Stage 2.
# We load it directly from its Hugging Face repository.
if use_cuda:
    preference_base_model = AutoModelForCausalLM.from_pretrained(
        HF_REPO_INSTRUCTION_MERGED,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map="auto",
        trust_remote_code=True,
    )

    preference_base_model = prepare_model_for_kbit_training(preference_base_model)

else:
    # CPU fallback. Training on CPU will be slow.
    preference_base_model = AutoModelForCausalLM.from_pretrained(
        HF_REPO_INSTRUCTION_MERGED,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

preference_base_model.config.use_cache = False

# Create a new LoRA adapter for preference tuning (Stage 3).
preference_lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=config.lora_r,
    lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

preference_model = get_peft_model(
    preference_base_model,
    preference_lora_config,
)

preference_model.print_trainable_parameters()

config.json:   0%|          | 0.00/725 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


In [107]:
# ============================================================
# 30. Configure DPO training
# ============================================================

import os
#import inspect
#from transformers import TrainingArguments
from trl import DPOTrainer
from trl import DPOConfig

try:
    from trl import DPOConfig
    has_dpo_config = True
except ImportError:
    DPOConfig = None
    has_dpo_config = False

preference_output_dir = "/content/pharma_tinyllama_preference_dpo_output"
preference_adapter_dir = "/content/pharma_tinyllama_preference_dpo_lora_adapter"

os.makedirs(preference_output_dir, exist_ok=True)
os.makedirs(preference_adapter_dir, exist_ok=True)

In [ ]:
# dpo_eval_mode = "steps" if len(preference_dataset["validation"]) > 0 else "no"

# # DPO-specific settings.
# # beta controls how strongly the model is pushed toward chosen answers over rejected answers.
# dpo_config_kwargs = dict(
#     output_dir=preference_output_dir,
#     num_train_epochs=3,
#     max_steps=5,
#     per_device_train_batch_size=1,
#     per_device_eval_batch_size=1,
#     gradient_accumulation_steps=8,
#     learning_rate=5e-5,
#     warmup_steps=5,
#     weight_decay=0.01,
#     logging_steps=1,
#     logging_first_step=True,
#     eval_steps=1,
#     save_steps=5,
#     save_total_limit=2,
#     fp16=False,
#     bf16=False,
#     report_to="none",
#     remove_unused_columns=False,

#     # DPO hyperparameters.
#     beta=0.1,
#     max_length=512,
#     max_prompt_length=256,
# )

# if has_dpo_config:
#     # Newer TRL versions use DPOConfig.
#     dpo_config_params = inspect.signature(DPOConfig.__init__).parameters

#     if "eval_strategy" in dpo_config_params:
#         dpo_config_kwargs["eval_strategy"] = dpo_eval_mode
#     elif "evaluation_strategy" in dpo_config_params:
#         dpo_config_kwargs["evaluation_strategy"] = dpo_eval_mode

#     safe_dpo_config_kwargs = {
#         key: value
#         for key, value in dpo_config_kwargs.items()
#         if key in dpo_config_params
#     }

#     removed_dpo_args = set(dpo_config_kwargs.keys()) - set(safe_dpo_config_kwargs.keys())
#     if removed_dpo_args:
#         print("Removed unsupported DPOConfig arguments:", removed_dpo_args)

#     dpo_training_args = DPOConfig(**safe_dpo_config_kwargs)

# else:
#     # Older TRL versions may use TrainingArguments.
#     training_args_params = inspect.signature(TrainingArguments.__init__).parameters

#     training_kwargs = {
#         key: value
#         for key, value in dpo_config_kwargs.items()
#         if key in training_args_params
#     }

#     if "eval_strategy" in training_args_params:
#         training_kwargs["eval_strategy"] = dpo_eval_mode
#     elif "evaluation_strategy" in training_args_params:
#         training_kwargs["evaluation_strategy"] = dpo_eval_mode

#     dpo_training_args = TrainingArguments(**training_kwargs)

# print(dpo_training_args)


In [ ]:
# # ============================================================
# # 31. Build DPOTrainer
# # ============================================================

# import inspect
# from trl import DPOTrainer

# dpo_trainer_kwargs = dict(
#     model=preference_model,
#     ref_model=None,
#     args=dpo_training_args,
#     train_dataset=preference_dataset["train"],
#     eval_dataset=preference_dataset["validation"] if len(preference_dataset["validation"]) > 0 else None,
# )

# dpo_trainer_params = inspect.signature(DPOTrainer.__init__).parameters

# # TRL versions differ: newer versions use processing_class, older versions use tokenizer.
# if "processing_class" in dpo_trainer_params:
#     dpo_trainer_kwargs["processing_class"] = tokenizer
# elif "tokenizer" in dpo_trainer_params:
#     dpo_trainer_kwargs["tokenizer"] = tokenizer

# # Older TRL versions may expect these DPO parameters directly in the trainer.
# if "beta" in dpo_trainer_params:
#     dpo_trainer_kwargs["beta"] = 0.1
# if "max_length" in dpo_trainer_params:
#     dpo_trainer_kwargs["max_length"] = 512
# if "max_prompt_length" in dpo_trainer_params:
#     dpo_trainer_kwargs["max_prompt_length"] = 256

# safe_dpo_trainer_kwargs = {
#     key: value
#     for key, value in dpo_trainer_kwargs.items()
#     if key in dpo_trainer_params
# }

# removed_trainer_args = set(dpo_trainer_kwargs.keys()) - set(safe_dpo_trainer_kwargs.keys())
# if removed_trainer_args:
#     print("Removed unsupported DPOTrainer arguments:", removed_trainer_args)

# dpo_trainer = DPOTrainer(**safe_dpo_trainer_kwargs)

# print("DPOTrainer is ready.")

### DPO training arguments

In [108]:
# ============================================================
# 30. Create DPO training arguments - Simple Version
# ============================================================

from trl import DPOConfig

dpo_training_args = DPOConfig(
    output_dir=preference_output_dir,

    # Training duration
    num_train_epochs=3,
    max_steps=5,

    # Batch settings
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    # Optimizer settings
    learning_rate=5e-5,
    warmup_steps=2,
    weight_decay=0.01,

    # Logging and evaluation
    logging_steps=1,
    logging_first_step=True,
    eval_strategy="steps",
    eval_steps=1,

    # Checkpoint saving
    save_steps=5,
    save_total_limit=2,

    # Precision settings
    fp16=False,
    bf16=False,

    # Disable external logging tools
    report_to="none",

    # Keep required columns
    remove_unused_columns=False,

    # DPO hyperparameter
    beta=0.1,
)

print(dpo_training_args)

DPOConfig(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
activation_offloading=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
beta=0.1,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
dataset_num_proc=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_dropout=True,
disable_tqdm=False,
discopop_tau=0.05,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat

### Build DPOTrainer

In [109]:
# ============================================================
# 31. Build DPOTrainer - Simple Student Version
# ============================================================

from trl import DPOTrainer

dpo_trainer = DPOTrainer(
    model=preference_model,
    ref_model=None,  # None means TRL will internally use the reference behavior
    args=dpo_training_args,

    train_dataset=preference_dataset["train"],
    eval_dataset=preference_dataset["validation"],

    processing_class=tokenizer,
)

print("DPOTrainer is ready.")

Adding EOS to train dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

DPOTrainer is ready.


### Start DPO preference tuning

In [110]:
# ============================================================
# 32. Start DPO preference tuning
# ============================================================

dpo_train_result = dpo_trainer.train()

print("DPO preference tuning completed.")
print(dpo_train_result)


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
1,0.693147,0.693147,1.764767,1413.000000,-3.204724,-3.393024,0.634932,0.000000,0.000000,0.000000,0.000000,-109.625213,-125.663549
2,0.693147,0.633934,1.759617,2789.000000,-3.196675,-3.387693,0.633461,0.015390,-0.108659,1.000000,0.124048,-109.471313,-126.750134
3,0.638961,0.482846,1.745855,4051.000000,-3.178007,-3.374614,0.627368,0.059350,-0.442483,1.000000,0.501833,-109.031712,-130.088375
4,0.475968,0.398659,1.735971,5380.000000,-3.164184,-3.364276,0.627368,0.091678,-0.677827,1.000000,0.769505,-108.708437,-132.441820
5,0.396036,0.361562,1.731433,6698.000000,-3.157097,-3.359216,0.627368,0.106993,-0.794957,1.000000,0.901951,-108.555281,-133.613124


DPO preference tuning completed.
TrainOutput(global_step=5, training_loss=0.5794518113136291, metrics={'train_runtime': 39.8684, 'train_samples_per_second': 1.003, 'train_steps_per_second': 0.125, 'total_flos': 49278084096000.0, 'train_loss': 0.5794518113136291, 'epoch': 1.0})


## DPO Training Metrics

| Parameter | Short Meaning |
|------------|------------|
| **Step** | Current optimizer step during training. |
| **Training Loss** | DPO loss on the training data; lower is generally better. |
| **Validation Loss** | DPO loss on unseen validation data; helps check generalization. |
| **Entropy** | Measures how uncertain the model is; higher means more random, lower means more confident. |
| **Num Tokens** | Total number of tokens processed so far. |
| **Logits/chosen** | Raw model score for the preferred answer. |
| **Logits/rejected** | Raw model score for the rejected answer. |
| **Mean Token Accuracy** | Average token-level prediction accuracy. |
| **Rewards/chosen** | DPO implicit reward for the preferred answer; should be higher. |
| **Rewards/rejected** | DPO implicit reward for the rejected answer; should be lower. |
| **Rewards/accuracies** | How often the model ranks the chosen answer above the rejected answer. |
| **Rewards/margins** | Difference between chosen reward and rejected reward; positive is good. |
| **Logps/chosen** | Log probability of the chosen answer; less negative means more likely. |
| **Logps/rejected** | Log probability of the rejected answer; ideally more negative than the chosen answer. |

---

## Simple Summary

In DPO training, the main goal is to make the model assign **higher probability** and **higher reward** to the **chosen answer** than to the **rejected answer**.

### Save DPO preference-tuned LoRA adapter

In [111]:
# ============================================================
# 33. Save DPO preference-tuned LoRA adapter
# ============================================================

dpo_trainer.model.save_pretrained(preference_adapter_dir)
tokenizer.save_pretrained(preference_adapter_dir)

print(f"Preference-tuned LoRA adapter saved to: {preference_adapter_dir}")
print(os.listdir(preference_adapter_dir))


Preference-tuned LoRA adapter saved to: /content/pharma_tinyllama_preference_dpo_lora_adapter
['adapter_model.safetensors', 'tokenizer_config.json', 'adapter_config.json', 'README.md', 'tokenizer.json', 'ref']


### Push Stage 3 DPO LoRA adapter to Hugging Face

In [113]:
# ============================================================
# Push Stage 3 DPO LoRA adapter to Hugging Face
# ============================================================

dpo_trainer.model.push_to_hub(
    HF_REPO_DPO_ADAPTER,
    private=False
)

tokenizer.push_to_hub(
    HF_REPO_DPO_ADAPTER,
    private=False
)

print("Stage 3 DPO LoRA adapter pushed to:")
print(HF_REPO_DPO_ADAPTER)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 34.5kB / 50.5MB            

  ...adapter_model.safetensors:   2%|2         |  629kB / 25.3MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Stage 3 DPO LoRA adapter pushed to:
AreebAhmxd/pharma-tinyllama-1.1b-dpo-lora-adapter


### Inferencing

In [114]:
# ============================================================
# 34. Reload preference-tuned model for inference
# ============================================================

import gc
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

if use_cuda:
    preference_inference_base_model = AutoModelForCausalLM.from_pretrained(
        merged_instruction_model_dir,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map="auto",
        trust_remote_code=True,
    )
else:
    preference_inference_base_model = AutoModelForCausalLM.from_pretrained(
        merged_instruction_model_dir,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

preference_inference_model = PeftModel.from_pretrained(
    preference_inference_base_model,
    preference_adapter_dir,
)

preference_inference_model.eval()

print("Preference-tuned model loaded successfully for inference.")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Preference-tuned model loaded successfully for inference.


In [115]:
# ============================================================
# 35. Preference-tuned inference helper
# ============================================================

def build_preference_prompt(instruction, input_text=""):
    instruction = instruction.strip()
    input_text = input_text.strip()

    if input_text:
        return (
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input_text}\n\n"
            f"### Response:\n"
        )

    return (
        f"### Instruction:\n{instruction}\n\n"
        f"### Response:\n"
    )

In [116]:
def generate_preference_response(instruction, input_text="", max_new_tokens=150):
    prompt = build_preference_prompt(instruction, input_text)

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(preference_inference_model.device)

    with torch.no_grad():
        outputs = preference_inference_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [117]:
# ============================================================
# 36. Test preference-tuned pharma model
# ============================================================

preference_test_questions = [
    "Explain the primary mechanism of action of metformin.",
    "Why should AI predictions in drug discovery be experimentally validated?",
    "Define pharmacovigilance.",
    "Explain why pharmacovigilance continues after drug approval.",
]

for question in preference_test_questions:
    print("=" * 100)
    print("QUESTION:")
    print(question)

    print("\nMODEL RESPONSE:")
    print(generate_preference_response(question, max_new_tokens=150))


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION:
Explain the primary mechanism of action of metformin.

MODEL RESPONSE:


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Explain the primary mechanism of action of metformin.

### Response:
Metformin reduces glucose production through inhibition of insulin receptor and AMP-activated protein kinase signaling, thereby reducing hepatic gluconeogenesis.

### Test Your Knowledge: Metformin Mechanism

1. Metformin is a sulfonylurea that acts by inhibiting glucose uptake through activation of insulin receptor or AMP-activated protein kinase signaling.
2. Metformin lowers blood glucose levels through decreased gluconeogenesis.
3. Metformin increases insulin sensitivity by inhibiting glucagon release and glucagon-
QUESTION:
Why should AI predictions in drug discovery be experimentally validated?

MODEL RESPONSE:


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Why should AI predictions in drug discovery be experimentally validated?

### Response:
Experimentally validated AI-driven prediction models should have been validated and replicated by human experts. Validation methods include data science, machine learning evaluation, and clinical trial analysis. AI-guided drug discovery is more likely to achieve positive results if it can be validated through human expert review of its performance on real-world datasets and biomarkers.

### Explain why AI prediction models for drug target validation are important.

### Response:
AI prediction models for drug target validation can identify novel targets or predict targets that are less likely to be functional targets. AI prediction models may also recommend drugs to optimize pharmacokinetic properties, cellular u
QUESTION:
Define pharmacovigilance.

MODEL RESPONSE:


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Define pharmacovigilance.

### Response:
Pharmacovigilance is the process of monitoring the safety and efficacy of a drug or medical device, including postmarketing studies and real-world evidence. Pharmacovigilance can inform regulatory decision-making about whether to approve a new indication, update an existing indication, or withdraw a drug from the market.

### Tip: Monitoring Safety

Evaluate adverse events after a drug is approved. Monitor for drug-related risks over time in clinical trials and real-world settings. Consider postapproval safety surveillance when developing new uses.

### Reference:

American Society of Clinical Oncology
QUESTION:
Explain why pharmacovigilance continues after drug approval.

MODEL RESPONSE:
### Instruction:
Explain why pharmacovigilance continues after drug approval.

### Response:
Pharmacovigilance activities continue after drug approval to monitor safety signals, identify adverse events, and evaluate the efficacy of therapy.

##

### Merge DPO preference adapter into the instruction-tuned base model

In [118]:
# ============================================================
# 37. Optional: Merge DPO preference adapter into the instruction-tuned base model
# ============================================================
# Use this only after preference tuning is complete and you want a standalone final model.

import os
import gc
import torch

from transformers import AutoModelForCausalLM
from peft import PeftModel

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

final_merged_preference_model_dir = "/content/pharma_tinyllama_final_preference_merged_model"
os.makedirs(final_merged_preference_model_dir, exist_ok=True)

In [119]:
# Load the merged instruction model in normal precision for safe merging.
base_model_for_preference_merge = AutoModelForCausalLM.from_pretrained(
    merged_instruction_model_dir,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)

# Attach the DPO preference LoRA adapter.
model_with_preference_adapter = PeftModel.from_pretrained(
    base_model_for_preference_merge,
    preference_adapter_dir,
)

# Merge the preference adapter into the instruction-tuned base model.
final_merged_preference_model = model_with_preference_adapter.merge_and_unload()

# Save final standalone model and tokenizer.
final_merged_preference_model.save_pretrained(final_merged_preference_model_dir)
tokenizer.save_pretrained(final_merged_preference_model_dir)

print(f"Final merged preference-tuned model saved to: {final_merged_preference_model_dir}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final merged preference-tuned model saved to: /content/pharma_tinyllama_final_preference_merged_model


In [120]:
# ============================================================
# 38. Push Final Merged DPO Model to Hugging Face Hub
# ============================================================

print(f"Pushing final merged DPO model to: {HF_REPO_DPO_MERGED}...")

# Push the standalone model weights
final_merged_preference_model.push_to_hub(
    HF_REPO_DPO_MERGED,
    private=False,
    commit_message="Pushing final pharma-domain DPO-tuned assistant model"
)

# Push the tokenizer
tokenizer.push_to_hub(
    HF_REPO_DPO_MERGED,
    private=False,
    commit_message="Add tokenizer for final pharma DPO-tuned model"
)

print(f"Successfully pushed to: https://huggingface.co/{HF_REPO_DPO_MERGED}")

Pushing final merged DPO model to: AreebAhmxd/pharma-tinyllama-1.1b-dpo-model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...q04fdu5/model.safetensors:   4%|3         | 80.0MB / 2.20GB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Successfully pushed to: https://huggingface.co/AreebAhmxd/pharma-tinyllama-1.1b-dpo-model
